# Rainfall Analysis and Planting Advisory

**Author:** Rosalind Nduta Muturi
**Cohort:** 1 Million Devs, Advanced Data Analytics with Python

**Scenario:** A farmer-support NGO sends SMS advice on planting windows in five counties. This notebook cleans a hand-typed Machakos station record, analyses monthly rainfall with NumPy, writes `rainfall_summary.csv` and ends with a short planting advisory.

## 1. Setup and raw data

In [1]:
import csv

import numpy as np

counties = ["Nairobi", "Nakuru", "Kisumu", "Mombasa", "Machakos"]

months = ["January", "February", "March", "April", "May", "June",
          "July", "August", "September", "October", "November", "December"]

clean_rows = [
    [64, 56, 92, 219, 176, 35, 18, 24, 31, 61, 165, 104],   # Nairobi
    [35, 42, 70, 145, 120, 78, 96, 105, 72, 55, 60, 38],    # Nakuru
    [58, 90, 150, 240, 180, 95, 70, 85, 90, 88, 130, 95],   # Kisumu
    [30, 15, 55, 150, 240, 110, 90, 65, 60, 90, 100, 70],   # Mombasa
]

machakos_raw = ["48", "35", "88", "N/A", "140", " 20 ",
                "9", "12", "", "45", "182", "96"]

print("Raw Machakos readings:", machakos_raw)

Raw Machakos readings: ['48', '35', '88', 'N/A', '140', ' 20 ', '9', '12', '', '45', '182', '96']


## 2. Clean the Machakos data

`int()` raises a `ValueError` for text such as `"N/A"` and for an empty string, so the function catches that specific exception. Values with stray spaces such as `" 20 "` convert fine because `int()` ignores surrounding whitespace. Negative rainfall is impossible, so it is also treated as invalid. Invalid months are filled with the rounded mean of the valid readings.

In [2]:
def clean_rainfall(raw_values, month_names):
    """Convert raw text readings to ints; impute invalid ones with the rounded mean of valid ones.

    Returns (cleaned_list, imputed_month_names, imputed_value).
    """
    parsed = []
    for value in raw_values:
        try:
            number = int(value)          # handles " 20 " but not "N/A" or ""
            if number < 0:
                raise ValueError("rainfall cannot be negative")
            parsed.append(number)
        except ValueError:
            parsed.append(None)          # mark as invalid for now

    valid = [v for v in parsed if v is not None]
    fill_value = int(round(sum(valid) / len(valid)))

    cleaned = [fill_value if v is None else v for v in parsed]
    imputed_months = [m for m, v in zip(month_names, parsed) if v is None]
    return cleaned, imputed_months, fill_value


machakos_clean, imputed_months, fill_value = clean_rainfall(machakos_raw, months)

print("Cleaned Machakos readings:", machakos_clean)
print(f"Imputed months: {', '.join(imputed_months)}")
print(f"Imputed value (rounded mean of valid readings): {fill_value} mm")

Cleaned Machakos readings: [48, 35, 88, 68, 140, 20, 9, 12, 68, 45, 182, 96]
Imputed months: April, September
Imputed value (rounded mean of valid readings): 68 mm


## 3. Combine into a 5 × 12 NumPy array

In [3]:
rainfall = np.array(clean_rows + [machakos_clean])

print("Shape:", rainfall.shape)
print("dtype:", rainfall.dtype)
assert rainfall.shape == (5, 12)

Shape: (5, 12)
dtype: int64


## 4. Annual totals and monthly averages

In [4]:
annual_totals = rainfall.sum(axis=1)        # one total per county (row)
monthly_avg = rainfall.mean(axis=0)         # one average per month (column), across counties

print("Annual total rainfall (mm):")
for county, total in zip(counties, annual_totals):
    print(f"  {county:<9} {total}")

print("\nAverage rainfall per month across all counties (mm):")
for month, avg in zip(months, monthly_avg):
    print(f"  {month:<9} {avg:.1f}")

Annual total rainfall (mm):
  Nairobi   1045
  Nakuru    916
  Kisumu    1371
  Mombasa   1075
  Machakos  811

Average rainfall per month across all counties (mm):
  January   47.0
  February  47.6
  March     91.0
  April     164.4
  May       171.2
  June      67.6
  July      56.6
  August    58.2
  September 64.2
  October   67.8
  November  127.4
  December  80.6


## 5. Wettest month per county

In [5]:
wettest_idx = np.argmax(rainfall, axis=1)
wettest_months = np.array(months)[wettest_idx]
wettest_values = rainfall.max(axis=1)

for county, month, mm in zip(counties, wettest_months, wettest_values):
    print(f"{county:<9} wettest month: {month} ({mm} mm)")

Nairobi   wettest month: April (219 mm)
Nakuru    wettest month: April (145 mm)
Kisumu    wettest month: April (240 mm)
Mombasa   wettest month: May (240 mm)
Machakos  wettest month: November (182 mm)


## 6. Dry months (below 50 mm)

In [6]:
dry_mask = rainfall < 50                    # boolean array, same shape as rainfall
dry_counts = dry_mask.sum(axis=1)           # True counts as 1

for county, n in zip(counties, dry_counts):
    print(f"{county:<9} dry months: {n}")

Nairobi   dry months: 4
Nakuru    dry months: 3
Kisumu    dry months: 0
Mombasa   dry months: 2
Machakos  dry months: 6


## 7. Best two-month planting window

`rainfall[:, :-1] + rainfall[:, 1:]` adds each month to the next one, so column 0 holds January + February, column 1 holds February + March, and so on (11 columns). The column with the largest sum marks the start of the wettest consecutive pair.

In [7]:
pair_sums = rainfall[:, :-1] + rainfall[:, 1:]        # shape (5, 11)
best_start = np.argmax(pair_sums, axis=1)
best_sum = pair_sums.max(axis=1)

months_arr = np.array(months)
window_labels = np.char.add(np.char.add(months_arr[best_start], "-"), months_arr[best_start + 1])

print("Pair-sum array shape:", pair_sums.shape)
for county, window, mm in zip(counties, window_labels, best_sum):
    print(f"{county:<9} planting window: {window} ({mm} mm combined)")

Pair-sum array shape: (5, 11)
Nairobi   planting window: April-May (395 mm combined)
Nakuru    planting window: April-May (265 mm combined)
Kisumu    planting window: April-May (420 mm combined)
Mombasa   planting window: April-May (390 mm combined)
Machakos  planting window: November-December (278 mm combined)


## 8. Write `rainfall_summary.csv`

In [8]:
header = ["county", "annual_total_mm", "wettest_month", "dry_months", "planting_window"]

with open("rainfall_summary.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(header)
    for row in zip(counties, annual_totals.tolist(), wettest_months.tolist(),
                   dry_counts.tolist(), window_labels.tolist()):
        writer.writerow(row)

# Read it back to confirm it is readable and has 5 data rows
with open("rainfall_summary.csv", newline="") as f:
    rows = list(csv.reader(f))

print(f"Rows written (excluding header): {len(rows) - 1}")
for row in rows:
    print(row)

Rows written (excluding header): 5
['county', 'annual_total_mm', 'wettest_month', 'dry_months', 'planting_window']
['Nairobi', '1045', 'April', '4', 'April-May']
['Nakuru', '916', 'April', '3', 'April-May']
['Kisumu', '1371', 'April', '0', 'April-May']
['Mombasa', '1075', 'May', '2', 'April-May']
['Machakos', '811', 'November', '6', 'November-December']


## Planting Advisory

Farmers in Nairobi, Nakuru, Kisumu and Mombasa should plan around the long rains, because April–May is the wettest two-month window in all four counties (Kisumu 420 mm, Nairobi 395 mm, Mombasa 390 mm and Nakuru 265 mm combined). Kisumu is the wettest county overall (1,371 mm a year) and has no dry months, so it is the most reliable place to plant on schedule. Machakos is much drier, with six months below 50 mm, so its farmers should target the short rains in November–December (278 mm combined) instead of April–May. SMS reminders should go out a week or two before each window opens, and farmers should confirm that rain has actually started before sowing. These windows come from average monthly totals, so they show when rain is most likely rather than guaranteeing it.

## Data Quality Notes

- **Machakos was hand-typed.** Two of its twelve readings were invalid (`"N/A"` for April and an empty string for September) and were replaced with the average of the ten valid readings, 68 mm. These two months are estimates, not measurements.
- **April matters for Machakos.** If the true April rainfall was much higher than 68 mm (roughly 138 mm or more), the April–May pair would beat November–December and the Machakos recommendation would change, so it should be checked against the station's original records.
- **One year of monthly totals.** The dataset has no year-to-year variation and no daily data, so it cannot show when rain starts or how reliable each season is.
- **Whitespace was harmless.** The `" 20 "` reading for June converted cleanly to 20 and was not imputed.